In [ ]:
!pip install anthropic pillow pandas -q

import anthropic
import base64
import json
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive
from sklearn.metrics import cohen_kappa_score

drive.mount('/content/drive')

IMAGE_FOLDER = '/content/drive/MyDrive/CDT_Dataset/images'
LABEL_FOLDER = '/content/drive/MyDrive/CDT_Dataset/labels'

SAVE_PATH = '/content/drive/MyDrive/CDT_Dataset/labels/ccss_labels_v2.json'

client = anthropic.Anthropic(api_key="YOUR_API_KEY_HERE")
print('Done!')

In [ ]:
SYSTEM_PROMPT = """You are a trained clinical rater specializing in the Clock Drawing Test (CDT).
Your role is to score a handwritten clock image objectively and consistently,
following the exact rubric below. You do NOT diagnose dementia — you only score.

SCORING RUBRIC (target time: 11:10)

CRITERION 1 — digit_count (0–2)
  Count only integers 1 through 12.
  2 = All 12 digits present, no duplicates, no extra numbers
  1 = 1 or 2 digits missing OR 1 or 2 extra/duplicate digits
  0 = 3 or more digits missing OR extra numbers beyond 12 present

CRITERION 2 — worst_quadrant_error (0–2)
  Identify the quadrant where digit placement is most incorrect.
  Count errors in THAT quadrant only.
  Error = digit in wrong position, illegible digit, or missing digit.
  2 = 0 errors in the worst quadrant
  1 = 1–2 errors in the worst quadrant
  0 = 3 or more errors in the worst quadrant

CRITERION 3 — spatial_arrangement (0–2)
  2 = Digits inside circle AND in correct clockwise sequence
  1 = Digits inside circle BUT counterclockwise OR partially wrong order
  0 = Gross distortion — digits outside circle, in a line, no circle, unrecognizable

CRITERION 4 — hands_present (0–2)
  2 = Two distinct hands; minute hand visibly longer; both meet at clear center
  1 = Two hands BUT lengths nearly identical OR no clear center OR one ambiguous
  0 = Only one hand, no hands, or perseveration

CRITERION 5 — hands_placement (0–2)
  Target: hour hand near 11, minute hand near 2. Tolerance: ±15 minutes of arc.
  2 = Both hands within tolerance
  1 = Exactly one hand correct
  0 = Both wrong OR time written as text OR pull-to-number error

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CRITICAL — IMAGE ORIENTATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The image may be rotated or flipped due to scanning.
BEFORE scoring, mentally rotate the image to find the
most natural clock orientation where:
- 12 is at the top
- Numbers increase clockwise
- The drawing makes most sense as a clock

Score based on the CORRECTED orientation only.
Do NOT penalize for image rotation — it is a scanning artifact,
not a patient error.

Examples:
- Image rotated 90°: rotate mentally, then score normally
- Image upside down: flip mentally, then score normally
- Unclear which way is up: choose orientation where
  digits 12, 3, 6, 9 are closest to top/right/bottom/left

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EDGE CASES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
- Two attempts: score the MORE COMPLETE drawing only
- Faint lines: score what is clearly visible
- Roman numerals: treat as valid equivalents
- Unidentifiable marks: ignore them
- Blurry image: set confidence below 0.5
- Small drawings: do NOT penalize for size alone,
  zoom in mentally and score what is visible

OUTPUT FORMAT — return ONLY this JSON, nothing else, no markdown fences:
{
  "image_id": "<filename>",
  "scores": {
    "digit_count": <0|1|2>,
    "worst_quadrant_error": <0|1|2>,
    "spatial_arrangement": <0|1|2>,
    "hands_present": <0|1|2>,
    "hands_placement": <0|1|2>
  },
  "total_score": <integer 0-10>,
  "risk_flag": <true if total_score < 6, else false>,
  "confidence": <float 0.0-1.0>,
  "flags": ["rotated_image"],
  "notes": "<max 2 sentences>",
  "reviewer": "llm"
}

Rules:
- total_score MUST equal sum of five criteria
- If image appears rotated, add "rotated_image" to flags
- Do NOT add any text before or after the JSON
- Do NOT include markdown fences"""

print('Load System Prompt Done!')

In [ ]:
from PIL import Image
import io
import time

def score_clock_image(image_path: str, debug=False) -> dict:
    img_path = Path(image_path)

    with Image.open(img_path) as img:
        img = img.convert('RGB')
        img.thumbnail((800, 800))
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=85)
        buffer.seek(0)
        b64 = base64.standard_b64encode(buffer.read()).decode('utf-8')

    for attempt in range(3):
        response = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=512,
            system=SYSTEM_PROMPT,
            messages=[{
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'source': {
                            'type': 'base64',
                            'media_type': 'image/jpeg',
                            'data': b64
                        }
                    },
                    {
                        'type': 'text',
                        'text': f'Score this clock drawing.\nimage_id: {img_path.name}\nReturn JSON only.'
                    }
                ]
            }]
        )

        raw = response.content[0].text.strip()

        if debug:
            print(f'Raw response: {repr(raw[:200])}')

        if not raw:
            print(f'  Empty response attempt {attempt+1}/3, retrying...')
            time.sleep(2)
            continue

        raw = raw.replace('```json', '').replace('```', '').strip()

        import re
        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())

        print(f'  No JSON found attempt {attempt+1}/3, retrying...')
        time.sleep(2)

    raise ValueError(f'Failed after 3 attempts. Last response: {repr(raw)}')

print('Load Finish')

In [ ]:
from PIL import ImageOps

def score_with_best_orientation(image_path: str) -> dict:
    """ลองหมุน 3 ทิศที่เหลือ เลือกที่ได้คะแนนสูงสุด"""
    img_path = Path(image_path)

    with Image.open(img_path) as img:
        img = img.convert('RGB')
        img = ImageOps.exif_transpose(img)
        img.thumbnail((800, 800))

        best_result = None
        best_score = -1

        for angle in [90, 180, 270]:  # ไม่ต้องทำ 0° แล้ว
            rotated = img.rotate(angle, expand=True)

            buffer = io.BytesIO()
            rotated.save(buffer, format='JPEG', quality=95)
            buffer.seek(0)
            b64 = base64.standard_b64encode(
                buffer.read()).decode('utf-8')

            try:
                response = client.messages.create(
                    model='claude-haiku-4-5-20251001',
                    max_tokens=512,
                    system=SYSTEM_PROMPT,
                    messages=[{
                        'role': 'user',
                        'content': [
                            {
                                'type': 'image',
                                'source': {
                                    'type': 'base64',
                                    'media_type': 'image/jpeg',
                                    'data': b64
                                }
                            },
                            {
                                'type': 'text',
                                'text': f'Score this clock drawing.\n'
                                        f'image_id: {img_path.name}\n'
                                        f'Return JSON only.'
                            }
                        ]
                    }]
                )

                raw = response.content[0].text.strip()
                raw = raw.replace('```json','').replace('```','').strip()

                import re
                json_match = re.search(r'\{.*\}', raw, re.DOTALL)
                if not json_match:
                    continue

                result = json.loads(json_match.group())
                score = result.get('total_score', 0)

                print(f'    {angle}°: score={score}/10  '
                      f'confidence={result.get("confidence",0):.2f}')

                if score > best_score:
                    best_score = score
                    best_result = result
                    best_result['rotation_used'] = angle

            except Exception as e:
                print(f'    {angle}° error: {e}')
                continue

    return best_result

print('Load func rotation finish')

In [ ]:
from PIL import Image, ImageEnhance
import io

def preprocess_image(img_path):
    """เตรียมรูปให้ Claude อ่านได้ดีขึ้น"""
    with Image.open(img_path) as img:
        img = img.convert('RGB')

        w, h = img.size

        # ถ้ารูปเล็กเกินไป ให้ขยายขึ้นแทนที่จะลดลง
        min_size = 600
        if w < min_size or h < min_size:
            scale = min_size / min(w, h)
            new_w = int(w * scale)
            new_h = int(h * scale)
            img = img.resize((new_w, new_h), Image.LANCZOS)
            print(f'  📐 ขยายรูปจาก {w}x{h} → {new_w}x{new_h}')

        # ถ้ารูปใหญ่เกินไป ให้ลดลง
        max_size = 1200
        if w > max_size or h > max_size:
            img.thumbnail((max_size, max_size), Image.LANCZOS)

        # เพิ่ม contrast นิดหน่อย ช่วยให้เห็นชัดขึ้น
        enhancer = ImageEnhance.Contrast(img)
        img = enhancer.enhance(1.3)

        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=95)  # quality สูงขึ้น
        buffer.seek(0)
        return base64.standard_b64encode(buffer.read()).decode('utf-8')

In [ ]:
import random

# สุ่ม 20 รูปจาก dataset
all_images = sorted(Path(IMAGE_FOLDER).glob('*.tif'))
test_images = random.sample(all_images, 20)

print(f'ทดสอบ {len(test_images)} รูป...\n')

llm_results = []

for img_path in test_images:
    try:
        result = score_clock_image(str(img_path))
        llm_results.append(result)
        print(f'{img_path.name}: {result["total_score"]}/10  '
              f'confidence={result["confidence"]:.2f}  '
              f'flags={result["flags"]}')
    except Exception as e:
        print(f'Error {img_path.name}: {e}')

print(f'\nDone {len(llm_results)}/20 รูป')

In [ ]:
# ทดสอบรูปที่แนบมา
test_path = f'{IMAGE_FOLDER}/40009711.tif'
result = score_clock_image(test_path, debug=True)

print(f'\nผลลัพธ์:')
print(f'digit_count: {result["scores"]["digit_count"]}')
print(f'worst_quadrant: {result["scores"]["worst_quadrant_error"]}')
print(f'spatial: {result["scores"]["spatial_arrangement"]}')
print(f'hands_present: {result["scores"]["hands_present"]}')
print(f'hands_placement: {result["scores"]["hands_placement"]}')
print(f'total: {result["total_score"]}/10')
print(f'confidence: {result["confidence"]}')
print(f'notes: {result["notes"]}')

In [ ]:
import random
import json

# โหลด label v1 ที่มีอยู่แล้ว
with open('/content/drive/MyDrive/CDT_Dataset/labels/ccss_labels.json') as f:
    labels_v1 = json.load(f)

# สุ่ม 20 รูปจากที่ label v1 แล้ว
sample_files = random.sample(list(labels_v1.keys()), 20)
print('20 รูปที่จะเปรียบ:')
for f in sample_files:
    print(f'  {f}')

In [ ]:
labels_v2 = {}

for filename in sample_files:
    img_path = f'{IMAGE_FOLDER}/{filename}'
    try:
        result = score_clock_image(img_path)
        labels_v2[filename] = result
        print(f'{filename}: {result["total_score"]}/10')
    except Exception as e:
        print(f'Error {filename}: {e}')

In [ ]:
from IPython.display import display
from PIL import Image

for filename in sample_files:
    img_path = f'{IMAGE_FOLDER}/{filename}'
    print(f'\n{"="*50}')
    print(f'ไฟล์: {filename}')
    print(f'v1 ให้: {labels_v1[filename].get("total","?")} /10')
    if filename in labels_v2:
        print(f'v2 ให้: {labels_v2[filename]["total_score"]} /10')
    img = Image.open(img_path)
    img.thumbnail((400, 400))
    display(img)
    input('Press Enter to next photo')

In [ ]:
human_labels = {
    '30002614.tif': [1, 2, 2, 1, 2],
    '30000741.tif': [2, 2, 2, 2, 2],
    '20005080.tif': [2, 0, 0, 2, 2],
    '40016030.tif': [0, 0, 0, 1, 2],
    '30000601.tif': [2, 2, 2, 2, 2],
    '30001964.tif': [2, 2, 2, 1, 2],
    '40013984.tif': [2, 1, 2, 1, 2],
    '30000256.tif': [2, 1, 2, 1, 2],
    '10009142.tif': [2, 2, 2, 2, 2],
    '30000012.tif': [2, 2, 2, 2, 2],
    '20005764.tif': [2, 2, 2, 2, 2],
    '10005863.tif': [2, 2, 2, 1, 2],
    '40015934.tif': [2, 1, 1, 1, 2],
    '30003703.tif': [2, 1, 2, 1, 2],
    '30002946.tif': [0, 0, 0, 1, 1],
    '10001413.tif': [2, 1, 2, 0, 1],
    '10007451.tif': [2, 2, 2, 1, 2],
    '40013953.tif': [2, 2, 2, 0, 1],
    '40013979.tif': [2, 2, 2, 1, 2],
    '40011352.tif': [2, 2, 2, 0, 1]
}

In [ ]:
# ทดสอบรูปที่ตะแคง เปรียบ 0° vs หมุน
test_path = f'{IMAGE_FOLDER}/30001964.tif'

# รัน 0° ก่อน
print('\nทิศปกติ (0°):')
result_0 = score_clock_image(test_path, debug=False)
print(f'  score={result_0["total_score"]}/10  confidence={result_0["confidence"]:.2f}')
print(f'  scores={result_0["scores"]}')
print(f'  notes={result_0["notes"]}')

# รัน 3 ทิศที่เหลือ
print('\nลองหมุน:')
result_rotated = score_with_best_orientation(test_path)

print('\n' + '='*50)
print(f'  0°:   {result_0["total_score"]}/10')
if result_rotated:
    print(f'  best: {result_rotated["total_score"]}/10  (หมุน {result_rotated["rotation_used"]}°)')

    if result_rotated['total_score'] > result_0['total_score']:
        diff = result_rotated['total_score'] - result_0['total_score']
        print(f'\nหมุนแล้วดีขึ้น {diff} คะแนน')
    else:
        print(f'\nหมุนแล้วไม่ดีขึ้น')
print('='*50)

In [ ]:
import numpy as np

domains = ['digit_count', 'worst_quadrant_error',
           'spatial_arrangement', 'hands_present', 'hands_placement']
domain_keys_v1 = ['A', 'B', 'C', 'D', 'E']

print('='*60)
print(f'{"Domain":<25} {"v1 MAE":>8} {"v2 MAE":>8} {"ดีกว่า":>8}')
print('='*60)

for domain, key_v1 in zip(domains, domain_keys_v1):
    h, v1_scores, v2_scores = [], [], []

    for filename in sample_files:
        if filename not in human_labels:
            continue
        if filename not in labels_v1:
            continue
        if filename not in labels_v2:
            continue

        h.append(human_labels[filename][domain_keys_v1.index(key_v1)])
        v1_scores.append(labels_v1[filename].get(key_v1, 0))
        v2_scores.append(labels_v2[filename]['scores'][domain])

    if not h:
        continue

    mae_v1 = np.mean(np.abs(np.array(h) - np.array(v1_scores)))
    mae_v2 = np.mean(np.abs(np.array(h) - np.array(v2_scores)))
    winner = 'v2' if mae_v2 < mae_v1 else 'v1' if mae_v1 < mae_v2 else 'เท่ากัน'

    print(f'{domain:<25} {mae_v1:>8.3f} {mae_v2:>8.3f} {winner:>8}')

print('='*60)

# คะแนนรวม
h_total = [sum(human_labels[f]) for f in sample_files if f in human_labels]
v1_total = [labels_v1[f].get('total', 0) for f in sample_files if f in human_labels and f in labels_v1]
v2_total = [labels_v2[f]['total_score'] for f in sample_files if f in human_labels and f in labels_v2]

mae_v1_total = np.mean(np.abs(np.array(h_total) - np.array(v1_total)))
mae_v2_total = np.mean(np.abs(np.array(h_total) - np.array(v2_total)))

print(f'\nMAE คะแนนรวม:')
print(f'  v1: {mae_v1_total:.3f}')
print(f'  v2: {mae_v2_total:.3f}')

if mae_v2_total < mae_v1_total:
    print('\nprompt v2 label is the best')
else:
    print('\nprompt v1 is the best')

In [ ]:
import os
from tqdm import tqdm

BATCH_SAVE = 10
SAVE_PATH = '/content/drive/MyDrive/CDT_Dataset/labels/ccss_labels_v2.json'

if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH, 'r') as f:
        labels_v2_full = json.load(f)
    print(f'โหลด progress เดิม: {len(labels_v2_full)} รูป')
else:
    labels_v2_full = {}
    print('Start Label')

all_images = [
    f for f in sorted(Path(IMAGE_FOLDER).glob('*.tif'))
    if f.name not in labels_v2_full
]

print(f'เหลือรูปที่ต้อง label: {len(all_images)} รูป')
print(f'ค่าใช้จ่ายโดยประมาณ: ${len(all_images) * 0.002:.2f}')

In [ ]:
errors = []
rotation_fixed = 0

for i, img_path in enumerate(tqdm(all_images)):
    try:
        # 0°
        result = score_clock_image(str(img_path))

        # ถ้าคะแนนต่ำ ลองหมุน
        if result['total_score'] <= 4:
            tqdm.write(f'\n  {img_path.name}: {result["total_score"]}/10 ลองหมุน...')
            result_rot = score_with_best_orientation(str(img_path))

            if result_rot and result_rot['total_score'] > result['total_score']:
                tqdm.write(f'  หมุน {result_rot["rotation_used"]}°: '
                          f'{result["total_score"]} → {result_rot["total_score"]}')
                result = result_rot
                rotation_fixed += 1
            else:
                tqdm.write(f'  หมุนแล้วไม่ดีขึ้น ใช้ 0° เดิม')
                result['rotation_used'] = 0
        else:
            result['rotation_used'] = 0

        # บันทึกลง dict
        labels_v2_full[img_path.name] = result

        # save ทุก 10 รูป
        if (i + 1) % BATCH_SAVE == 0:
            with open(SAVE_PATH, 'w') as f:
                json.dump(labels_v2_full, f, ensure_ascii=False, indent=2)
            tqdm.write(f'บันทึก: {len(labels_v2_full)} รูป '
                      f'(แก้ rotation {rotation_fixed} รูป)')

    except Exception as e:
        errors.append(img_path.name)
        tqdm.write(f'Error: {img_path.name} — {e}')
        continue

# บันทึกครั้งสุดท้าย
with open(SAVE_PATH, 'w') as f:
    json.dump(labels_v2_full, f, ensure_ascii=False, indent=2)

print(f'\nDone!')
print(f'label สำเร็จ : {len(labels_v2_full)} รูป')
print(f'แก้ rotation  : {rotation_fixed} รูป')
print(f'Error        : {len(errors)} รูป')
if errors:
    print(f'รูปที่ error  : {errors[:10]}')